# 02 · Campaign — generate across lengths × secondary-structure biases

**Standard slot:** *design campaign.* **For Project 03 this means:** the core generative campaign —
hundreds of RFdiffusion monomer backbones across lengths {80,120,200,300} and SS-biases {α,β,mixed},
each redesigned with ProteinMPNN (8 seqs) and refolded for self-consistency, written to
`results/backbones.csv` (D2). **Diversity before filtering.**

> **Compute reality.** This runs **anywhere on the `mock` backend** (no GPU). The *real* campaign:
> a free **T4 handles small batches only**; the full lengths×topology sweep (100s of backbones,
> lengths up to 300) realistically needs an **A100 / HPC**. Batch overnight; do not fight a T4.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify cell (pinned upstreams — tools change)

These tools move; pin a commit/tag and HTTP-check the upstreams still exist before a run. Replace the
`# pinned:` comments with the exact commit/tag you used and log them in `LOG.md`.

In [ ]:
import requests

PINNED_UPSTREAMS = {
    # tool          : (url, "pinned commit/tag — REPLACE with the one you used")
    "RFdiffusion":  ("https://github.com/RosettaCommons/RFdiffusion", "pin a commit/tag"),
    "ColabDesign":  ("https://github.com/sokrypton/ColabDesign",      "pin a commit/tag"),
    "Foldseek":     ("https://github.com/steineggerlab/foldseek",     "pin a release"),
}
for tool, (url, pin) in PINNED_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=20)
        print(f"  {tool:12s} {r.status_code}  {url}  (pinned: {pin})")
    except Exception as e:
        print(f"  {tool:12s} CHECK FAILED ({e!r})  {url}")
print("\nIf any is not 200, the repo moved/renamed — update the install + the pin, and log it.")

## 1 · The campaign grid

Generate `N_PER_CELL` backbones in every (length × ss_bias) cell. On the `mock` backend keep it
small so the notebook is fast; on the real backend scale `N_PER_CELL` up (the catalog asks for
*hundreds* of backbones total) and move to A100/HPC.

In [ ]:
import rfdiff_tools as rt
import pandas as pd

LENGTHS  = rt.CAMPAIGN_LENGTHS      # (80, 120, 200, 300)
SS_BIAS  = rt.SS_BIASES             # ("alpha", "beta", "mixed")
N_PER_CELL = 8                      # mock: small & fast. Real: scale to dozens -> 100s total (A100/HPC).
TOOL_GEN   = "mock"                 # -> "rfdiffusion" on a GPU runtime
TOOL_FOLD  = "mock"                 # -> "esmfold" (triage) / "af2" (top picks)
TOOL_NOV   = "mock"                 # -> "foldseek"
SEED       = 0

print(f"grid: {len(LENGTHS)} lengths x {len(SS_BIAS)} topologies x {N_PER_CELL} = "
      f"{len(LENGTHS)*len(SS_BIAS)*N_PER_CELL} backbones (mock run)")

## 2 · Run: generate → ProteinMPNN → self-consistency → novelty

For each backbone: ProteinMPNN designs ~8 sequences, each is refolded (AF2/ESMFold), and we keep the
**best-of-8 scRMSD**; Foldseek/TM-align scores novelty against the PDB. The mock backend folds this
into one deterministic call per backbone so the loop runs anywhere — **the numbers are synthetic
(`EXAMPLE_DATA_`)**, encoding the realistic prior that short all-α folds best and long all-β worst.

In [ ]:
rows = []
for L in LENGTHS:
    for ss in SS_BIAS:
        bbs = rt.generate_backbones(length=L, ss_bias=ss, n=N_PER_CELL, tool=TOOL_GEN, seed=SEED)
        for bb in bbs:
            sc = rt.self_consistency(bb, tool=TOOL_FOLD, n_seqs=8)   # best-of-8 MPNN seqs
            nv = rt.novelty_tm(bb.pdb_path or bb.backbone_id, tool=TOOL_NOV)
            rows.append(dict(
                backbone_id=bb.backbone_id, length=L, ss_bias=ss, seed=bb.seed,
                scrmsd=sc.scrmsd, plddt=sc.plddt, tm_to_pdb=nv.tm_to_pdb,
                synthetic=bb.synthetic, tool_gen=bb.tool, tool_fold=sc.tool, tool_nov=nv.tool))

backbones = pd.DataFrame(rows)
backbones.to_csv("results/backbones.csv", index=False)
print("wrote results/backbones.csv", backbones.shape)
backbones.head()

## 3 · First look at the campaign

Foldability rate per cell (fraction with scRMSD < 2 Å). On the synthetic mock data you should see the
expected pattern: high for short all-α, collapsing toward long all-β. **This per-cell rate IS the
result** — notebook 04 turns it into the frontier.

In [ ]:
import rfdiff_tools as rt
foldable = backbones.assign(foldable=backbones.scrmsd < rt.SELF_CONSISTENT_SCRMSD)
rate = (foldable.groupby(["ss_bias", "length"])["foldable"].mean()
        .unstack("length").round(2))
print("EXAMPLE_DATA (synthetic) — foldable rate (scRMSD < 2 A) by topology x length:")
print(rate)
print("\nnovel (TM<0.5) overall:", int((backbones.tm_to_pdb < rt.NOVEL_TM).sum()),
      "/", len(backbones))

## 4 · The real RFdiffusion call (ColabDesign) + A100 note

On a GPU runtime, swap the mock backend for the real one. The pattern below shows the ColabDesign
RFdiffusion call `generate_backbones` wraps; see `scripts/rfdiff_tools.py` for the adapter TODOs.

In [ ]:
# Real backend (GPU runtime; A100/HPC for the full sweep). Illustrative — fill in 00_setup installs.
#
#   # via the ColabDesign RFdiffusion notebook (pin the commit):
#   ./run_inference.py 'contigmap.contigs=[80-80]' inference.num_designs=8 \
#       inference.output_prefix=results/backbones/len80_alpha \
#       'scaffoldguided.ss_bias=alpha'
#
# then in this notebook:
#   TOOL_GEN, TOOL_FOLD, TOOL_NOV = "rfdiffusion", "esmfold", "foldseek"
#   # ESMFold for triage across the whole pool; re-fold top picks with AF2 in notebook 04.
#
# COMPUTE: T4 OK for a single cell / small n. The full 4 lengths x 3 topologies x dozens
# realistically needs an A100 (Colab Pro+) or HPC. Batch overnight; log GPU + runtime per cell.
print("Real campaign: set TOOL_* to the real backends on A100/HPC; batch the full grid overnight.")

## D2 checklist
- [ ] `results/backbones.csv`: hundreds of backbones across the full lengths × SS-bias grid (real run), one row per backbone with length, ss_bias, scrmsd, plddt, tm_to_pdb, seed.
- [ ] Design log: every config (length, ss_bias, num_designs), MPNN settings (temp, seqs/backbone), seeds, GPU, runtimes in `LOG.md`.
- [ ] Version-verify cell run; pinned commits recorded.
- [ ] Per-cell foldability rate inspected (don't leave under-sampled cells — esp. long all-β).
- [ ] 3–4 page interim report on the early frontier shape.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on your backbone pool.